# Лекция 06. Алгоритмы сортировки

Сортировка — это не олимпиадный трюк, а обычная операция: расставить платежи в очереди, показать свежие операции первыми или объединить два журнала событий.

## Цели

После лекции вы сможете:

- формулировать ключ и направление сортировки;
- использовать `sorted()`, `.sort()` и стабильность;
- реализовывать сортировку вставками, слиянием, heapsort и radix sort;
- объяснять идею и оценки quicksort;
- отличать сортировки сравнениями от поразрядной сортировки;
- объяснять, почему bogosort корректен, но практически бесполезен;
- выбирать алгоритм по входным данным и ограничениям;
- проверять, что сортировка действительно корректна.

## Перед началом

Нужны списки, словари, функции, циклы и оценки сложности из предыдущих занятий. Все примеры работают на обычном Python без сторонних библиотек.

## Сначала задаём порядок

Службе выплат нужно обработать платежи: больший приоритет раньше, а при одинаковом приоритете — более ранняя заявка раньше.

До выбора алгоритма надо ответить на три вопроса:

1. Что сортируем? Записи о платежах.
2. По какому ключу? По паре `(-priority, created_at)`.
3. Что делать с равными ключами? Сохранить исходный порядок.

In [ ]:
payments = [
    {"id": "p-101", "priority": 1, "created_at": "09:10"},
    {"id": "p-102", "priority": 3, "created_at": "10:00"},
    {"id": "p-103", "priority": 3, "created_at": "08:45"},
    {"id": "p-104", "priority": 1, "created_at": "09:10"},
]

ordered = sorted(
    payments,
    key=lambda payment: (-payment["priority"], payment["created_at"]),
)
print([payment["id"] for payment in ordered])

## `sorted()` и `.sort()`

`sorted(iterable)` создаёт новый список. Метод `list.sort()` переставляет элементы исходного списка и возвращает `None`.

В прикладной функции чаще безопаснее `sorted()`: вызывающий код не обнаружит, что его данные внезапно изменились. `.sort()` уместен, когда список принадлежит текущему коду и копия не нужна.

In [ ]:
amounts = [5300, 1200, 8700]
new_amounts = sorted(amounts)
print(amounts, new_amounts)

result = amounts.sort()
print(amounts, result)

## Ключ сортировки

Параметр `key` не сравнивает две записи. Он один раз вычисляет для каждой записи значение, по которому Python будет её размещать.

Кортежи сравниваются слева направо, поэтому одним ключом удобно задавать несколько правил. Числовое поле можно сортировать в обратном направлении, поменяв знак. Для сложной логики лучше дать функции имя.

In [ ]:
def payment_order(payment):
    return (-payment["priority"], payment["created_at"])

ordered = sorted(payments, key=payment_order)
for payment in ordered:
    print(payment["priority"], payment["created_at"], payment["id"])

## Стабильность

Стабильный алгоритм не меняет взаимный порядок элементов с равными ключами. Это полезно не только в теории: заявки одного статуса останутся в порядке поступления.

Стабильность позволяет сортировать в несколько проходов. Сначала сортируем по второстепенному полю, затем стабильной сортировкой — по главному.

In [ ]:
requests = [
    {"id": "r-1", "status": "done", "manager": "Ира"},
    {"id": "r-2", "status": "new", "manager": "Олег"},
    {"id": "r-3", "status": "done", "manager": "Анна"},
    {"id": "r-4", "status": "new", "manager": "Борис"},
]
status_order = {"new": 0, "done": 1}

grouped = sorted(requests, key=lambda request: status_order[request["status"]])
print([(request["status"], request["id"]) for request in grouped])

In [ ]:
by_manager_then_status = sorted(requests, key=lambda request: request["manager"])
by_manager_then_status.sort(key=lambda request: status_order[request["status"]])
print([(request["status"], request["manager"]) for request in by_manager_then_status])

## Сортировка вставками

Представим небольшую очередь счетов, уже почти упорядоченную по сумме. Берём очередной элемент и вставляем его в правильное место отсортированного префикса.

Инвариант цикла: перед каждой итерацией `result[:i]` уже отсортирован. Пока предыдущий ключ **строго больше** текущего, сдвигаем предыдущий элемент вправо.

In [ ]:
def insertion_sort(records, key=lambda item: item):
    result = records.copy()
    for i in range(1, len(result)):
        current = result[i]
        j = i - 1
        while j >= 0 and key(result[j]) > key(current):
            result[j + 1] = result[j]
            j -= 1
        result[j + 1] = current
    return result

invoices = [
    {"id": "i-1", "amount": 1200},
    {"id": "i-2", "amount": 1800},
    {"id": "i-3", "amount": 1500},
    {"id": "i-4", "amount": 2100},
]
print([item["id"] for item in insertion_sort(invoices, key=lambda item: item["amount"])])

Если вход уже упорядочен, внутренний цикл почти не работает: время `Θ(N)`. Если порядок обратный, каждый новый элемент проходит через весь префикс: `Θ(N²)`. Дополнительная память сверх возвращаемой копии — `O(1)`.

Поэтому вставки хороши для маленьких или почти упорядоченных участков, но не для сотен тысяч случайных записей.

## Слияние двух журналов

Банк и платёжный шлюз уже отдают события по времени. Повторно сортировать всё необязательно: держим по указателю в каждом списке и каждый раз забираем более раннее событие.

Если времена равны, берём событие слева. Так слияние остаётся стабильным.

In [ ]:
def merge_events(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i]["timestamp"] <= right[j]["timestamp"]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

bank = [
    {"id": "bank-1", "timestamp": 10},
    {"id": "bank-2", "timestamp": 20},
]
gateway = [
    {"id": "gateway-1", "timestamp": 20},
    {"id": "gateway-2", "timestamp": 25},
]
print([event["id"] for event in merge_events(bank, gateway)])

## Сортировка слиянием

Идея merge sort состоит из знакомых действий:

1. разделить список пополам;
2. отсортировать каждую половину тем же способом;
3. линейно слить две отсортированные половины.

Глубина разбиения — порядка `log N`, на каждом уровне сливаются все `N` элементов. Поэтому время — `Θ(N log N)` во всех случаях. В приведённой реализации дополнительная память — `O(N)` плюс стек рекурсии.

In [ ]:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def merge_sort(values):
    if len(values) <= 1:
        return values.copy()
    middle = len(values) // 2
    return merge(merge_sort(values[:middle]), merge_sort(values[middle:]))

amounts = [5300, 1200, 8700, 1200, 4100]
print(merge_sort(amounts))

## Quicksort: идея без магии

Выбираем опорный элемент `pivot`, разделяем остальные элементы на меньшие и большие, затем повторяем для частей. После удачных делений задача примерно каждый раз уменьшается вдвое — в среднем получается `Θ(N log N)`.

Если опорный элемент постоянно оказывается крайним, одна часть почти пуста, другая короче всего на один элемент: худший случай `Θ(N²)`. Классический вариант обычно работает на месте, но не стабилен.

Quicksort важно уметь объяснить и трассировать. Для обычного Python-кода вручную писать его вместо `sorted()` почти никогда не нужно.

In [ ]:
amounts = [5300, 1200, 8700, 4100, 2300]
pivot = amounts[-1]
smaller = [amount for amount in amounts[:-1] if amount <= pivot]
greater = [amount for amount in amounts[:-1] if amount > pivot]
print("pivot:", pivot)
print("left:", smaller)
print("right:", greater)

## Heapsort: максимум всегда под рукой

Нужно упорядочить суммы счетов. Сначала перестраиваем список в **max-heap**: родитель не меньше своих детей. Для элемента с индексом `i` дети находятся по индексам `2 * i + 1` и `2 * i + 2`. Поэтому отдельное дерево и узлы нам не нужны — куча живёт прямо в списке.

Максимум оказывается в позиции `0`. Меняем его местами с последним элементом, исключаем последний элемент из кучи и восстанавливаем свойство кучи просеиванием вниз (`sift down`). Повторяем, пока куча не опустеет.

Построение кучи занимает `O(N)`, каждое из `N` извлечений — `O(log N)`, итого `O(N log N)` во всех случаях. Классический heapsort работает на месте с `O(1)` дополнительной памяти, но не стабилен и не ускоряется на уже упорядоченном входе.

In [ ]:
def sift_down(values, root, size):
    while True:
        left_child = 2 * root + 1
        if left_child >= size:
            return

        larger_child = left_child
        right_child = left_child + 1
        if right_child < size and values[right_child] > values[left_child]:
            larger_child = right_child

        if values[root] >= values[larger_child]:
            return

        values[root], values[larger_child] = values[larger_child], values[root]
        root = larger_child

def heap_sort(values):
    result = values.copy()
    size = len(result)

    for root in range(size // 2 - 1, -1, -1):
        sift_down(result, root, size)

    for end in range(size - 1, 0, -1):
        result[0], result[end] = result[end], result[0]
        sift_down(result, 0, end)

    return result

invoice_amounts = [5300, 1200, 8700, 4100, 2300]
print(heap_sort(invoice_amounts))
print(invoice_amounts)  # учебная функция вернула копию

В функции выше копия сделана ради удобного контракта «не менять вход». Сам алгоритм после создания `result` переставляет элементы на месте и использует только несколько переменных. Если сортировать исходный список, дополнительная память heapsort будет `O(1)`. Это и есть его заметное преимущество перед обычным merge sort.

## Почему постоянно возникает `N log N`

Сравнение отвечает только на вопрос вроде «левый элемент меньше правого?». Возможных порядков `N!`, а одно сравнение делит варианты максимум на две группы. Поэтому любая универсальная сортировка, основанная только на сравнениях, требует в худшем случае `Ω(N log N)` сравнений.

Линейные сортировки существуют для специальных ключей, например целых чисел из небольшого диапазона. Это уже другие предпосылки, а не нарушение нижней границы.

## Radix sort: сортируем не сравнениями

Пусть коды транзакций — неотрицательные целые числа. Можно несколько раз стабильно разложить их по цифрам: сначала по единицам, затем по десяткам, сотням и так далее. После каждого прохода числа собираются из корзин `0..9` в прежнем порядке внутри корзины.

Это **LSD radix sort**: начинаем с младшего разряда (`least significant digit`). Стабильность каждого прохода обязательна — иначе следующий разряд разрушит уже учтённый порядок младших цифр.

Для `D` разрядов и основания `B` время равно `Θ(D · (N + B))`, память — `O(N + B)`. При фиксированной длине машинного целого и фиксированном основании это линейно по `N`. Нижняя граница `Ω(N log N)` не нарушена: алгоритм использует структуру целочисленного ключа, а не только попарные сравнения.

In [ ]:
def radix_sort(values):
    if any(value < 0 for value in values):
        raise ValueError("radix_sort ожидает неотрицательные числа")

    result = values.copy()
    if not result:
        return result

    place = 1
    maximum = max(result)
    while maximum // place > 0:
        buckets = [[] for _ in range(10)]
        for value in result:
            digit = value // place % 10
            buckets[digit].append(value)
        result = [value for bucket in buckets for value in bucket]
        place *= 10

    return result

transaction_codes = [170, 45, 75, 90, 802, 24, 2, 66]
print(radix_sort(transaction_codes))
print(transaction_codes)

Учебная реализация работает только с неотрицательными целыми числами. Для отрицательных значений, строк переменной длины или составных ключей нужны дополнительные правила. В прикладном Python чистая реализация radix sort часто проиграет `sorted()`: встроенная сортировка написана на C и не ограничивает форму ключа.

## Bogosort: алгоритм как антипример

Bogosort проверяет порядок и, если список не отсортирован, случайно перемешивает его целиком. Для `N` различных элементов вероятность получить нужную перестановку за одну попытку равна `1 / N!`. Ожидается около `N!` перемешиваний, каждое перемешивание и проверка занимают `Θ(N)`, поэтому ожидаемое время — `Θ(N · N!)`. Конечной верхней границы времени нет: неудачные перемешивания могут продолжаться сколько угодно.

Практического применения у bogosort нет. Он полезен как проверка понимания: корректность результата ещё не делает алгоритм приемлемым. В демонстрации ниже есть лимит попыток, поэтому ноутбук гарантированно завершится. Функция возвращает копию ради безопасного контракта; само перемешивание списка выполняется на месте.

In [ ]:
from random import Random

def is_sorted(values):
    return all(left <= right for left, right in zip(values, values[1:]))

def bogosort_demo(values, max_shuffles=10_000, seed=42):
    result = values.copy()
    random = Random(seed)
    shuffles = 0

    while not is_sorted(result):
        if shuffles >= max_shuffles:
            raise RuntimeError("bogosort превысил безопасный лимит")
        random.shuffle(result)
        shuffles += 1

    return result, shuffles

result, shuffles = bogosort_demo([3, 1, 2])
print(result)
print("Перемешиваний:", shuffles)

## Что делает Python

`sorted()` и `.sort()` используют Timsort — стабильную адаптивную сортировку. Она находит в данных уже упорядоченные серии (`runs`), доводит короткие серии и эффективно сливает их. Внутри сочетаются идеи вставок и слияния, но промышленная реализация заметно аккуратнее учебных функций выше.

Практические свойства:

- худший случай `O(N log N)`;
- уже упорядоченный вход обрабатывается за `O(N)`;
- используется существующий порядок в частично отсортированных данных;
- ключ вычисляется один раз для каждого элемента;
- сортировка стабильна.

Для прикладной задачи правильный выбор по умолчанию — встроенная сортировка.

## Сравнение алгоритмов

| Алгоритм | Лучший случай | Средний / худший | Доп. память | Стабильность | Где уместен |
|---|---:|---:|---:|---|---|
| Вставки | `Θ(N)` | `Θ(N²)` / `Θ(N²)` | `O(1)` | да при строгом сравнении | маленькие и почти готовые участки |
| Слияние | `Θ(N log N)` | `Θ(N log N)` | `O(N)` | да при правильном равенстве | гарантированное время, потоки и внешние данные |
| Quicksort | `Θ(N log N)` | `Θ(N log N)` / `Θ(N²)` | зависит от реализации | обычно нет | системные реализации и обучение |
| Heapsort | `Θ(N log N)` | `Θ(N log N)` | `O(1)` на месте | нет | гарантированное время при жёстком лимите памяти |
| Radix sort | `Θ(D · (N + B))` | `Θ(D · (N + B))` | `O(N + B)` | да при стабильных проходах | целочисленные ключи известного формата |
| Bogosort | `Θ(N)` | ожидаемо `Θ(N · N!)` / без границы | `O(1)` на месте | нет | только учебный антипример |
| Timsort в Python | `Θ(N)` | `O(N log N)` | до `O(N)` | да | обычный прикладной код |

## Как проверять сортировку

Одного красивого примера мало. Проверяем четыре свойства:

1. ключи соседних элементов не убывают;
2. ни один элемент не потерян и не добавлен;
3. равные ключи сохранили порядок, если обещана стабильность;
4. исходный список не изменился, если функция должна вернуть копию.

Отдельные случаи: пустой список, один элемент, повторы, уже готовый и обратный порядок.

In [ ]:
for case in [[], [7], [2, 2, 1], [1, 2, 3], [3, 2, 1]]:
    original = case.copy()
    assert merge_sort(case) == sorted(case)
    assert heap_sort(case) == sorted(case)
    assert radix_sort(case) == sorted(case)
    assert case == original

assert bogosort_demo([1, 2, 3]) == ([1, 2, 3], 0)
print("Базовые проверки пройдены")

## Неожиданно, но по правилам

Ниже не ошибки интерпретатора, а следствия контрактов и свойств разобранных алгоритмов.

### 1. После `items = items.sort()` в переменной лежит `None`

Методы, меняющие изменяемый контейнер на месте, обычно ничего полезного не возвращают. Список уже изменён; присваивать результат метода не надо.

In [ ]:
amounts = [300, 100, 200]
returned = amounts.sort()
print(amounts)
print(returned is None)

### 2. Обратная сортировка тоже стабильна

`reverse=True` меняет направление ключей, но записи с одинаковым ключом остаются в исходном взаимном порядке.

In [ ]:
payments_same_amount = [
    {"id": "first", "amount": 500},
    {"id": "second", "amount": 700},
    {"id": "third", "amount": 500},
]
result = sorted(payments_same_amount, key=lambda item: item["amount"], reverse=True)
print([item["id"] for item in result])

### 3. Ключ вызывается не на каждое сравнение

Python заранее вычисляет ключ ровно один раз для каждой записи. Поэтому журнал вызовов ниже содержит столько элементов, сколько было платежей, хотя сравнений могло быть больше.

In [ ]:
calls = []

def amount_key(payment):
    calls.append(payment["id"])
    return payment["amount"]

sorted(payments_same_amount, key=amount_key)
print(calls)
print(len(calls) == len(payments_same_amount))

### 4. Куча не выглядит отсортированной

В max-heap каждый родитель не меньше своих детей, но между соседями и разными ветвями полного порядка нет. Поэтому корректная куча ниже не убывает слева направо — и это нормально.

In [ ]:
heap = [5300, 1200, 8700, 4100, 2300]
for root in range(len(heap) // 2 - 1, -1, -1):
    sift_down(heap, root, len(heap))

print(heap)
print("Максимум в корне:", heap[0])
print("По убыванию?", heap == sorted(heap, reverse=True))

### 5. Radix sort может быть линейным

Это не опровержение границы `Ω(N log N)`. Граница относится к универсальным сортировкам, которые узнают порядок только через сравнения. Radix sort смотрит на отдельные разряды ключа и потому требует дополнительных предпосылок о данных.

### 6. Лучший случай bogosort — линейный

Если список уже отсортирован, алгоритм один раз проверит соседние элементы и не выполнит ни одного перемешивания. Хороший лучший случай ничего не говорит о типичном поведении: ожидаемое время на произвольной перестановке остаётся факториальным.

In [ ]:
ordered, shuffles = bogosort_demo([1, 2, 3, 4])
print(ordered)
print("Перемешиваний:", shuffles)

## Самопроверка

1. Почему для порядка «приоритет убывает, время возрастает» подходит ключ `(-priority, created_at)`?
2. Чем `sorted(data)` отличается от `data.sort()`?
3. Какое сравнение сохраняет стабильность при вставке и слиянии?
4. Когда вставки работают за линейное время?
5. Почему максимум в max-heap находится в позиции `0`?
6. Из-за чего quicksort деградирует до квадратичного времени?
7. Почему проходы radix sort должны быть стабильными?
8. Почему линейный radix sort не нарушает границу `Ω(N log N)`?
9. Почему линейный лучший случай не спасает bogosort?
10. Какие свойства результата надо тестировать помимо порядка?

## Источники

- [Python Sorting HOWTO](https://docs.python.org/3/howto/sorting.html) — ключи, стабильность, несколько проходов и Timsort.
- [CPython `listsort.txt`](https://github.com/python/cpython/blob/main/Objects/listsort.txt) — устройство встроенной сортировки из первых рук.
- [Документация `sorted()`](https://docs.python.org/3/library/functions.html#sorted) — контракт встроенной функции.

## Итоги

- Сортировка начинается с точного определения ключа, направления и правила равенства.
- Стабильность сохраняет порядок равных элементов и позволяет сортировать в несколько проходов.
- Вставки хороши на маленьких почти готовых данных; слияние даёт стабильные `Θ(N log N)`; heapsort гарантирует `Θ(N log N)` с `O(1)` памяти, но теряет стабильность; quicksort зависит от качества разбиений.
- Radix sort использует структуру целочисленного ключа и при фиксированном числе разрядов работает линейно; это не универсальная сортировка сравнениями.
- Bogosort показывает, что корректность без приемлемой оценки времени недостаточна.
- В обычной программе используем встроенный стабильный и адаптивный Timsort.
- Собственный алгоритм нужен прежде всего для понимания, специальных ограничений или собеседования — не для замены одной строки `sorted()`.